<a href="https://colab.research.google.com/github/BahruzHuseynov/Portfolio/blob/main/Deep_Learning_Projects/CNN_Architecture/CNN1_LeNet5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LeNet-5

<img src = "https://raw.githubusercontent.com/valoxe/image-storage-1/master/research-paper-summary/lenet-5/2.png?token=AMAXSKIFVOFJARUUJULNGWS6WMEQG">

<img src = "https://www.philschmid.de/static/blog/getting-started-with-cnn-by-calculating-lenet-layer-manually/lenet-5.svg">

In [89]:
import torch
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader

from torch import nn
import torch.nn.functional as F
from torch import optim

In [90]:
applied_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [91]:
trainset = torchvision.datasets.MNIST(root='./data', train=True,
                                      download=True, transform = applied_transform)
testset = torchvision.datasets.MNIST(root='./data', train=False,
                                     download=True, transform = applied_transform)

In [92]:
trainset

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=[0.5], std=[0.5])
           )

In [93]:
len(trainset), len(testset)

(60000, 10000)

In [94]:
sample_x, sample_y = trainset[0]
print(sample_x.size(), "-->" , sample_y)

torch.Size([1, 28, 28]) --> 5


In [95]:
set(trainset.targets.numpy())

{0, 1, 2, 3, 4, 5, 6, 7, 8, 9}

In [96]:
train_loader = DataLoader(trainset, batch_size = 64, shuffle = True)
test_loader = DataLoader(testset, batch_size = 16, shuffle = True)

## Model Building

In [97]:
class LeNet5_1(nn.Module):
    def __init__(self, num_classes, in_channels = 1):
        super(LeNet5_1, self).__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(in_channels = in_channels, out_channels = 6, kernel_size = 5, stride = 1),
            nn.Tanh(),
            nn.AvgPool2d(kernel_size = 2, stride = 2),
            nn.Tanh(),
            nn.Conv2d(in_channels = 6, out_channels = 16, kernel_size = 5, stride = 1),
            nn.AvgPool2d(kernel_size = 2, stride = 2),
            nn.Tanh(),
            nn.Flatten(),
            nn.Linear(in_features = 16*4*4, out_features = 120),
            nn.Tanh(),
            nn.Linear(in_features = 120, out_features = 84),
            nn.Tanh(),
            nn.Linear(in_features = 84, out_features = num_classes)
        )

    def forward(self, x):
        x = self.seq(x)
        return x

In [99]:
class LeNet5_2(nn.Module):
    def __init__(self, num_classes, in_channels = 1):
        super(LeNet5_2, self).__init__()

        self.seq = nn.Sequential(
            nn.Conv2d(in_channels = in_channels, out_channels = 6, kernel_size = 5, stride = 1),
            nn.Tanh(),
            nn.AvgPool2d(kernel_size = 2, stride = 2),
            nn.Conv2d(in_channels = 6, out_channels = 16, kernel_size = 5, stride = 1),
            nn.Tanh(),
            nn.AvgPool2d(kernel_size = 2, stride = 2),
        )

        self.classifier = nn.Sequential(
            nn.Linear(in_features=16*4*4, out_features=120),
            nn.Tanh(),
            nn.Linear(in_features=120, out_features=84),
            nn.Tanh(),
            nn.Linear(in_features=84, out_features=num_classes)
        )

    def forward(self, x):
        x = self.seq(x)
        x = torch.flatten(x, 1)  # batch_size, filters * width * height
        return self.classifier(x)

In [100]:
class LeNet5_3(nn.Module):
    def __init__(self, num_classes, in_channels = 1):
        super(LeNet5_3, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=6, kernel_size=5, stride=1)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1)
        self.fc1 = nn.Linear(in_features=16 * 4 * 4, out_features=120)
        self.fc2 = nn.Linear(in_features=120, out_features=84)
        self.fc3 = nn.Linear(in_features=84, out_features=num_classes)

    def forward(self, x):
        x = F.tanh(self.conv1(x))
        x = F.tanh(F.avg_pool2d(x, kernel_size=2, stride=2))
        x = F.tanh(self.conv2(x))
        x = F.tanh(F.avg_pool2d(x, kernel_size=2, stride=2))

        x = torch.flatten(x, 1)
        x = F.tanh(self.fc1(x))
        x = F.tanh(self.fc2(x))
        x = self.fc3(x)
        return x

In [103]:
model1 = LeNet5_1(10)
model2 = LeNet5_2(10)
model3 = LeNet5_3(10)

In [104]:
print(model1)
print("------------------------------------------------------")
print(model2)
print("------------------------------------------------------")
print(model3)

LeNet5_1(
  (seq): Sequential(
    (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
    (1): Tanh()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Tanh()
    (4): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (6): Tanh()
    (7): Flatten(start_dim=1, end_dim=-1)
    (8): Linear(in_features=256, out_features=120, bias=True)
    (9): Tanh()
    (10): Linear(in_features=120, out_features=84, bias=True)
    (11): Tanh()
    (12): Linear(in_features=84, out_features=10, bias=True)
  )
)
------------------------------------------------------
LeNet5_2(
  (seq): Sequential(
    (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
    (1): Tanh()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    (4): Tanh()
    (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
  )
  (classifier): Sequential(
    (0): Linear(in_features=256, out_features=120, bias

## Training

In [105]:
f_loss = nn.CrossEntropyLoss()
optimizer = optim.Adam(model2.parameters(), lr=0.001)

In [106]:
num_epochs = 3
for epoch in range(num_epochs):
    model2.train()
    train_loss = 0.0
    for batch_idx, (samples, targets) in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model2(samples)
        loss = f_loss(outputs, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    print("Epoch:", epoch + 1, "-->", train_loss)

Epoch: 1 --> 269.87674856185913
Epoch: 2 --> 82.70536659378558
Epoch: 3 --> 57.354644367005676


## Testing and Performance Check

In [110]:
from sklearn.metrics import classification_report

In [112]:
model1.eval()
preds = []
actual = []
test_loss = 0.0
with torch.no_grad():
    for samples, targets in test_loader:
        outputs = model2(samples)
        _, predicted = torch.max(outputs.data, 1)
        preds.extend(predicted.numpy())
        actual.extend(targets.numpy())

In [113]:
print(classification_report(actual, preds))

              precision    recall  f1-score   support

           0       0.98      0.99      0.99       980
           1       1.00      0.99      0.99      1135
           2       0.99      0.98      0.98      1032
           3       0.97      0.98      0.98      1010
           4       0.99      0.98      0.98       982
           5       0.97      0.99      0.98       892
           6       0.99      0.98      0.98       958
           7       0.97      0.99      0.98      1028
           8       0.99      0.96      0.98       974
           9       0.97      0.98      0.97      1009

    accuracy                           0.98     10000
   macro avg       0.98      0.98      0.98     10000
weighted avg       0.98      0.98      0.98     10000

